In [ ]:
# ============================================================
# Overfitting / Underfitting Check
# 过拟合/欠拟合检查
# Note: Since train and val sets are merged in the final models,
# we compare Train MSE vs Test MSE instead of Train vs Val.
# 注意：由于最终模型合并了训练集和验证集，
# 这里用训练集MSE与测试集MSE对比来判断过拟合/欠拟合。
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ── Diagnosis helper / 诊断辅助函数 ──────────────────────────
def diagnose_final(model_name, train_mse, test_mse, threshold_ratio=1.5):
    """
    Print an overfitting/underfitting diagnosis based on Test/Train MSE ratio.
    根据测试集与训练集MSE比值，打印过拟合/欠拟合诊断结果。

    Diagnosis rules / 诊断规则:
        Train MSE very high          → Underfitting / 训练误差极高     → 欠拟合
        Test/Train ratio > threshold → Overfitting  / 比值过大         → 过拟合
        Otherwise                    → Good fit     / 否则             → 拟合良好
    """
    ratio = test_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE       : {train_mse:,.4f}")
    print(f"  Test  MSE       : {test_mse:,.4f}")
    print(f"  Test/Train Ratio: {ratio:.3f}")

    if train_mse > 1e10:
        print("  ⚠ Diagnosis : UNDERFITTING — training error is very high")
        print("  ⚠ 诊断结果  : 欠拟合 — 训练误差过高，模型未能有效学习")
    elif ratio > threshold_ratio:
        print("  ⚠ Diagnosis : OVERFITTING  — test MSE is much larger than train MSE")
        print("  ⚠ 诊断结果  : 过拟合 — 测试集误差远大于训练集误差")
    else:
        print("  ✓ Diagnosis : GOOD FIT     — train and test MSE are close")
        print("  ✓ 诊断结果  : 拟合良好 — 训练集与测试集误差接近")
    print(f"{'='*55}")


# ============================================================
# MODEL 1 — Final Baseline
# 模型1 — 最终基线模型
#
# Reuse values already computed in the original code above.
# baseline_train_mse and baseline_test_mse_final were both
# calculated immediately after baseline_final_model was trained.
# 直接复用原代码中已经计算好的值。
# baseline_train_mse 和 baseline_test_mse_final 在
# baseline_final_model 训练完成后已经立即计算好了。
# ============================================================

m1_train_mse = baseline_train_mse        # already computed above / 原代码已算好
m1_test_mse  = baseline_test_mse_final   # already computed above / 原代码已算好

diagnose_final("Model 1 — Final Baseline", m1_train_mse, m1_test_mse)


# ============================================================
# MODEL 2 — Final Enhanced
# 模型2 — 最终增强模型（含邻域统计特征）
#
# Reuse values already computed in the original code above.
# enhanced_test_mse_final was calculated immediately after
# model_enh_final was trained.
# Train MSE was not computed in the original code,
# so we calculate it once here using the already-fitted model.
# 直接复用原代码中已经计算好的值。
# enhanced_test_mse_final 在 model_enh_final 训练完成后已经立即计算好了。
# 训练集MSE在原代码中未计算，因此这里用已训练好的模型计算一次。
# ============================================================

m2_test_mse  = enhanced_test_mse_final   # already computed above / 原代码已算好
m2_train_mse = mean_squared_error(       # not in original code, compute once here
    y_train_final,                       # 原代码未计算，在此计算一次
    model_enh_final.predict(X_train_enh_final)
)

diagnose_final("Model 2 — Final Enhanced", m2_train_mse, m2_test_mse)


# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Test MSE
# 可视化1 — 柱状图：训练集与测试集MSE对比
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
train_mses = [m1_train_mse, m2_train_mse]
test_mses  = [m1_test_mse,  m2_test_mse]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

# Plot train and test MSE bars side by side
# 并排绘制训练集和测试集MSE柱子
bars1 = ax.bar(x - width/2, train_mses, width,
               label='Train MSE', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, test_mses,  width,
               label='Test MSE',  color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact value
# 在柱子顶部标注具体数值
for bar, val in zip(bars1, train_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Learning curves (Train MSE vs Epoch)
# 可视化2 — 学习曲线
#
# Since there is no validation set after merging,
# we use the held-out test set as the reference line.
# 合并后无独立验证集，用测试集MSE作为参考线。
#
# How to interpret / 学习曲线解读:
#   Train curve falls and flattens, test line is close → Good fit
#   两线接近且收敛                                       → 拟合良好
#   Train curve keeps falling, far below test line     → Overfitting
#   训练线持续下降远低于测试线                            → 过拟合
#   Both stay high and flat                            → Underfitting
#   两线均居高不下                                       → 欠拟合
# ============================================================

def collect_train_curve(Xtr, ytr, Xte, yte, hidden, act, max_iter=800):
    """
    Train epoch-by-epoch using warm_start and record train/test MSE.
    使用 warm_start 逐epoch训练，记录每轮的训练集和测试集MSE。
    """
    # warm_start=True retains weights between calls so training continues
    # warm_start=True 保留上次权重，使每次 fit 从上次结束处继续训练
    m = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation=act,
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=1,        # One epoch per call / 每次只训练一个epoch
        warm_start=True,   # Continue from previous weights / 保留权重继续训练
        early_stopping=False,
        random_state=26
    )
    train_curve, test_curve = [], []
    for _ in range(max_iter):
        m.fit(Xtr, ytr)
        train_curve.append(mean_squared_error(ytr, m.predict(Xtr)))
        test_curve.append(mean_squared_error(yte, m.predict(Xte)))
    return train_curve, test_curve


print("\nCollecting learning curves (this may take a moment)…")
print("正在收集学习曲线数据（可能需要一点时间）…")

# Collect curves for both models
# 分别收集两个模型的学习曲线
lc_train_m1, lc_test_m1 = collect_train_curve(
    X_train_final_base, y_train_final_base,
    X_test_final_base,  y_test,
    hidden=(128, 32), act='tanh'
)
lc_train_m2, lc_test_m2 = collect_train_curve(
    X_train_enh_final, y_train_final,
    X_test_enh_final,  y_test,
    hidden=(128, 32), act='tanh'
)

# Plot side-by-side learning curves
# 并排绘制两个模型的学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lc_tr, lc_te, title in zip(
    axes,
    [lc_train_m1, lc_train_m2],
    [lc_test_m1,  lc_test_m2],
    ['Model 1 — Final Baseline', 'Model 2 — Final Enhanced']
):
    epochs = range(1, len(lc_tr) + 1)

    # Solid line = Train MSE, Dashed line = Test MSE
    # 实线 = 训练集MSE，虚线 = 测试集MSE
    ax.plot(epochs, lc_tr, label='Train MSE', color='steelblue')
    ax.plot(epochs, lc_te, label='Test MSE',  color='darkorange', linestyle='--')

    # Mark the epoch with lowest test MSE
    # 标注测试集MSE最低的epoch
    # 简单说就是：**在800轮训练中，测试集误差最低的那一轮是第几轮
    best_ep  = int(np.argmin(lc_te)) + 1
    best_mse = min(lc_te)
    ax.axvline(best_ep, color='red', linestyle=':', linewidth=1.2,
               label=f'Best Test MSE epoch={best_ep}')
    ax.scatter([best_ep], [best_mse], color='red', zorder=5)
    ax.annotate(
        f"Best\n{best_mse:.4f}",
        xy=(best_ep, best_mse),
        xytext=(best_ep + 30, best_mse + 0.01),
        arrowprops=dict(arrowstyle='->', color='red'),
        fontsize=8, color='red'
    )

    ax.set_title(title)
    ax.set_xlabel('Epoch / 训练轮数')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Learning Curves — Overfitting / Underfitting Check\n学习曲线 — 过拟合/欠拟合检查',
    fontsize=13
)
plt.tight_layout()
plt.show()

## Overfitting / Underfitting Analysis
## 过拟合 / 欠拟合分析


### 1. Why This Analysis / 为什么要做这个分析

Validating Model Improvement / 验证模型改进的真实性

The primary goal of this project is to demonstrate that adding
neighbourhood-based features improves prediction performance.
However, reporting Test MSE alone is insufficient. A lower Test MSE
could result from genuine generalisation improvement, or it could
simply be a consequence of the model overfitting the training data.
本项目的核心目标是证明加入邻域特征能够提升预测性能。然而，仅汇报测试集MSE是不够的。
更低的测试集MSE可能来自真实的泛化能力提升，也可能只是模型过拟合训练数据的结果。

By comparing Train MSE and Test MSE, we can confirm that the
improvement observed in Model 2 reflects genuine generalisation
rather than memorisation of the training data. If the Test/Train
ratio remains close to 1.0 for both models, we can conclude that
the performance gain is reliable and trustworthy.
通过对比训练集MSE和测试集MSE，我们可以确认模型2观察到的改进反映的是真实的泛化能力提升，而非对训练数据的过度记忆。
若两个模型的测试集/训练集MSE比值均接近1.0，则可以认为性能提升是可靠且可信的。

Overfitting and underfitting represent two common failure modes in
machine learning models. Overfitting occurs when a model learns the
training data too well, including its noise, and consequently
performs poorly on unseen data. Underfitting occurs when a model
lacks sufficient capacity to capture the underlying patterns in the
data, resulting in high error on both training and test sets.
过拟合和欠拟合是机器学习模型中两种常见的失败模式。过拟合发生在模型对训练数据（包括其噪声）学习过度，
导致在未见数据上表现较差的情况。欠拟合则发生在模型容量不足，无法捕捉数据中的潜在规律，导致训练集和测试集误差均较高的情况。




### 2. Diagnosis Method 

We define the **Test/Train MSE Ratio** as the diagnostic metric:
我们定义**测试集/训练集MSE比值**作为诊断指标：

 Condition / 条件 | Diagnosis / 诊断 
  Train MSE extremely high / 训练集MSE极高   Underfitting / 欠拟合 — model failed to learn / 模型未能学习 
  Ratio > 1.5   Overfitting / 过拟合 — poor generalisation / 泛化能力差  
  Ratio ≈ 1.0   Good fit / 拟合良好 — errors are close / 误差接近  

The threshold of **1.5** means the Test MSE is at most 50% larger
than Train MSE, which is considered acceptable for a model trained
on a finite dataset.
阈值**1.5**表示测试集MSE最多比训练集MSE大50%，对于有限数据集训练的模型来说这是可接受的范围。




### 3. Results / 结果

    Model / 模型 | Train MSE | Test MSE | Ratio | Diagnosis / 诊断 
  Model 1 — Baseline | 0.2398 | 0.2485 | 1.036 | ✓ Good Fit / 拟合良好 
  Model 2 — Enhanced | 0.1557 | 0.1805 | 1.159 | ✓ Good Fit / 拟合良好 

Both models show a ratio close to 1.0, indicating neither model is
overfitting or underfitting. Model 2 achieves a lower Test MSE than
Model 1, confirming that neighbourhood features improve generalisation
without introducing overfitting.
两个模型的比值均接近1.0，说明均未出现过拟合或欠拟合。
模型2的测试集MSE低于模型1，证明邻域特征在提升泛化性能的同时没有引入过拟合。




### 4. Learning Curve Analysis / 学习曲线分析

Beyond the final MSE values, the learning curves provide insight
into how each model behaves throughout the entire training process.
除最终MSE数值外，学习曲线还能反映每个模型在整个训练过程中的行为。

Specifically, the learning curves allow us to determine whether
800 training epochs are sufficient for convergence. If the curves
are still decreasing at epoch 800, additional training may further
improve performance. Conversely, if the test curve begins to rise
at an earlier epoch, that point represents the optimal stopping
point, and continued training beyond it would introduce overfitting.
具体而言，学习曲线能帮助我们判断800轮训练是否足以达到收敛。
若曲线在第800轮仍在下降，增加训练轮数可能进一步提升性能。
反之，若测试集曲线在某轮之后开始上升，该点即为最佳停止点，继续训练则会引入过拟合。

This analysis therefore serves both as a quality check on the
final models and as a diagnostic tool for future model refinement.
因此，本分析既是对最终模型的质量验证，也是未来模型改进的诊断工具。

  Pattern / 曲线形态 | Interpretation / 解读 
  Both lines decrease and converge / 两线下降并收敛  Good fit / 拟合良好 
  Test rises while Train keeps falling / 测试线上升训练线继续下降  Overfitting from that epoch / 从该点开始过拟合 
  Both lines stay high and flat / 两线居高不下  Underfitting / 欠拟合 

The **red vertical line** marks the epoch at which Test MSE is lowest.
红色竖线标注了测试集MSE最低的epoch。
- If near epoch 800 → model is still improving, no overfitting
  若接近第800轮 → 模型仍在持续改善，无过拟合
- If much earlier with Test MSE rising afterwards → overfitting present
  若出现在中间且之后测试集MSE上升 → 存在过拟合
